## pdf2image

In [ ]:
!pip install -q PyMuPDF Pillow

In [ ]:
import fitz               # PyMuPDF
from PIL import Image, ImageFilter
import os
import shutil

def pdf_to_jpg(pdf_path, output_folder="output_images", dpi=300,
               threshold=220, output_folder_bw="bw_imgs", pdf_id=0, blur_radius=0.5):
    """
    Convert each page of a PDF to:
      • color_image_<page>.jpg
      • bw_image_<page>.jpg

    Args:
      pdf_path (str): path to .pdf file
      output_folder (str): where to save JPEGs
      dpi (int): rendering resolution
      threshold (0–255): gray cutoff for B/W
      blur_radius (float): Gaussian blur radius on B/W
    """
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(output_folder_bw, exist_ok=True)
    doc = fitz.open(pdf_path)
    page_count = doc.page_count

    for i, page in enumerate(doc, start=1):
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # — Color JPEG
        color_file = os.path.join(output_folder, f"color_image_{i}.jpg")
        img.save(color_file, "JPEG", quality=100, dpi=(dpi, dpi))

        # — Grayscale → threshold → blur → B/W JPEG
        gray = img.convert("L")
        bw = gray.point(lambda x: 255 if x > threshold else 0)
        bw = bw.filter(ImageFilter.GaussianBlur(blur_radius))
        bw_file = os.path.join(output_folder_bw, f"bw_image_{pdf_id}_{i}.jpg")
        bw.save(bw_file, "JPEG", quality=100, dpi=(dpi, dpi))

    doc.close()
    print(f"Done! For pdf {pdf_id} Saved {page_count} page(s) to '{output_folder}'")

if __name__ == "__main__":

  # -- single pdf parse --
  # pdf_path = "/content/90-QP20-R-001_1_Process & Inst_Diagram _PID_Legend_and_Sybmols.pdf"
  # pdf_to_jpg(pdf_path,
  #             output_folder="/content/output_images")


  # -- iterate over pdf/img files --
  input_dir = "/content/dataset_09/Equipment patches/"
  output_bw = "/content/bw_images/"
  os.makedirs(output_bw, exist_ok=True)

  # -- iterate over dir --
  for idx, file_x in enumerate(os.listdir(input_dir)):

    # -- check if pdf --
    if (file_x.endswith(".pdf") or file_x.endswith(".PDF")):
      pdf_path = os.path.join(input_dir, file_x)
      pdf_to_jpg(pdf_path,
                output_folder=output_bw,
                pdf_id = idx,
                dir_id = dirIdx,
                output_folder_bw=output_bw)

    # -- check if image --
    elif (file_x.endswith(".jpg") or file_x.endswith(".JPG")):
      src_path = os.path.join(input_dir, file_x)
      dst_path = os.path.join(output_bw, file_x)
      print(src_path, dst_path)
      shutil.copy(src_path, dst_path)
      print(f"Done! For :{file_x} Copied to {dst_path}")

    # -- else skip --
    else:
      print("Not a pdf or jpeg file -- file {}".format(file_x))

## Data Preparation

### Raw Dataset
  - These code cells are only usable whenever creating
  a raw dataset from large images and its corresponding csv with same names. Which later on be handle by the labeling team for correcting mislabeling.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/images
!rm -rf /content/labels
!rm -rf /content/output_images/

In [ ]:
# slicing large images into patches of 1792x1792

import os
import cv2
import json
import math

# === CONFIG ===
input_image_dir = "/content/bw_images"             # Directory with large images
input_annotation_dir = "/content/Images and csv files"   # JSON or TXT files, same name as image
output_image_dir = "patch_images"
output_annotation_dir = "labels"
# os.makedirs(write_img_dir, exist_ok=True)
patch_size = 1792

os.makedirs(output_image_dir, exist_ok=True)
# os.makedirs(output_annotation_dir, exist_ok=True)

In [ ]:
# functions
import cv2
import json
import os


def adjust_labels(labels, x_offset, y_offset, patch_size=1792):
  """Adjust the label of large images into relative to the patch considering the patch_size

  Args:
    labels (list): list of labels : [[xmin, ymin, xmax, ymax],...]
    x_offset (int): offset of the image in x-axis
    y_offset (int): offset of the image in y-axis

  Returns: adjusted_labels : [[xmin, ymin, xmax, ymax],...]
  """
  # labels : [xmin, ymin, xmax, ymax]
  # x_offset = 1792, 3584, 5376
  # y_offset = 1792, 3584, 5376

  image_labels = []

  for l in labels:
    if ((x_offset <= l[1]) and (l[1] <= (x_offset + patch_size))): ##--> for horizontal parse
      if ((y_offset <= l[0]) and (l[0] <= (y_offset + patch_size))): ## --> for Vertical Parse
        image_labels.append(l)
        # print("x_offset :{} y_offset :{} label :{}".format(x_offset, y_offset, l))

  if len(image_labels) == 0:
    return

  for index in range(len(image_labels)):
    image_labels[index][0] -= y_offset
    image_labels[index][1] -= x_offset
    image_labels[index][2] -= y_offset
    image_labels[index][3] -= x_offset

    image_labels[index][2] = min(1792, image_labels[index][2])
    image_labels[index][3] = min(1792, image_labels[index][3])

  return image_labels



def xywh_to_yolo(labels, image_width, image_height):
  """Convert the labels from xywh to yolo

  Args:
    labels (list): list of labels : [[xcenter, ycenter, width, height], ...]
    image_width (int): width of the image
    image_height (int): height of the image

  Returns: labels (list): list of normalized labels : [[xcenter, ycenter, width, height], ...]
  """
  if labels is None or len(labels)==0: return
  image_labels = [] # holding data in format of yolo
  for index in range(len(labels)):
    x_center_norm = labels[index][0] / image_width
    y_center_norm = labels[index][1] / image_height
    width_norm = labels[index][2] / image_width
    height_norm = labels[index][3] / image_height

    image_labels.append([x_center_norm, y_center_norm, width_norm, height_norm])
  return image_labels



def yolo_to_xywh(labels, image_width, image_height):
  """Convert the labels from yolo to xywh normalized vales
  Args:
    labels (list): list of labels : [[xcenter, ycenter, width, height], ...]
    image_width (int): width of the image
    image_height (int): height of the image

  Return: labels (list): list of labels : [[xcenter, ycenter, width, height], ...]
  """
  if labels is None or len(labels)==0: return
  image_labels = [] # holding data in format of xywh
  for index in range(len(labels)):
    x_center = labels[index][0] * image_width
    y_center = labels[index][1] * image_height
    width = labels[index][2] * image_width
    height = labels[index][3] * image_height

    image_labels.append([x_center, y_center, width, height])
  return image_labels


def xyxy_to_xywh(labels):
  """Convert the labels from xyxy to xywh

  Args:
    labels (list): list of labels : [[xmin, ymin, xmax, ymax], ...]

  Returns:
    labels (list): list of labels : [[xcenter, ycenter, width, height], ...]
  """
  if labels is None or len(labels) == 0: return
  image_labels = [] # holding data in format of xcycwh
  for index in range(len(labels)):
    # labels[[xmin, ymin, xmax, ymax], ...]
    width = labels[index][2] - labels[index][0]   # width = x2 - x1
    height = labels[index][3] - labels[index][1]  # height = y2 - y1
    x_center = (labels[index][0] + labels[index][2])//2
    y_center = (labels[index][1] + labels[index][3])//2
    image_labels.append([x_center, y_center, width, height])
  return image_labels


def xywh_to_xyxy(labels):
  """Convert the labels from xywh to xyxy

  Args:
    labels (list): list of labels: [[xcenter, ycenter, width, height], ...]

  Returns:
    labels (list): list of labels : [[xmin, ymin, xmax, ymax], ...]
  """
  if labels is None or len(labels)==0: return
  image_labels = [] # holding data in format of XminYminXmaxYmax
  for index in range(len(labels)):
    x_min = labels[index][0] - labels[index][2]//2
    y_min = labels[index][1] - labels[index][3]//2
    x_max = labels[index][0] + labels[index][2]//2
    y_max = labels[index][1] + labels[index][3]//2

    image_labels.append([int(x_min), int(y_min), int(x_max), int(y_max)])
  return image_labels


def pad_patch(img, patch_size):
  """Pad the patch into multiple of patch size

  Args:
    img (np.array): image in RGB format
    patch_size (int): size of the patch

  Returns:
    img (np.array): padded image
  """
  h, w = img.shape[:2]

  # Calculate required padding
  h, w = img.shape[:2]
  pad_h = math.ceil(h / patch_size) * patch_size - h
  pad_w = math.ceil(w / patch_size) * patch_size - w

  # Pad the image (append zeros - black pixels)
  padded_img = cv2.copyMakeBorder(
      img,
      0, pad_h,  # top, bottom
      0, pad_w,  # left, right
      cv2.BORDER_CONSTANT,
      value=(255, 255, 255)  # black padding
  )
  return padded_img


def patch_images(img, file_name, *labels, patch_size=1792):

  if img is None: return

  #-- Configuration --#
  img = pad_patch(img, patch_size) ## pad the image
  h, w, _ = img.shape
  patches_h = h//patch_size
  patches_w = w//patch_size
  img_id = 0
  obj_id = 0


  #-- Splitting Images --#
  for idx_1, h_idx in enumerate(range(patches_h)): # 4
    for idx_2, w_idx in enumerate(range(patches_w)): # 6

      # calculate patch
      x = h_idx * patch_size
      y = w_idx * patch_size

      # create patch (patch_size)
      patch = img[x:x+patch_size, y:y+patch_size]

      # calculate patch base for adjustment of labels relative to patch
      h_base = 0 if h_idx==0 else patch_size * (h_idx)
      w_base = 0 if w_idx==0 else patch_size * (w_idx)
      # print(f"h_idx :{h_idx} w_idx :{w_idx} h_base :{h_base} w_base :{w_base} x:{x}-{x+patch_size} y:{y}-{y+patch_size} patch :{patch.shape}")

      # adjust the labels according to the yolo format
      adjusted_labels = adjust_labels(labels, x_offset=h_base, y_offset=w_base, patch_size=patch_size)
      adjusted_labels = xyxy_to_xywh(adjusted_labels)
      adjusted_labels = xywh_to_yolo(adjusted_labels, patch_size, patch_size)

      # # writing paths (patch, labels)
      patch_path = os.path.join(output_image_dir, f"{file_name[:-4]}_patch_{h_idx}_{w_idx}.jpg")
      labels_path = os.path.join(output_annotation_dir, f"{file_name[:-4]}_patch_{h_idx}_{w_idx}.txt")

      # write patch & labels
      cv2.imwrite(patch_path, patch)
      with open(labels_path, "w") as f:

        if adjusted_labels is not None:
          for label in adjusted_labels:
            f.write(f"{obj_id} {label[0]} {label[1]} {label[2]} {label[3]} \n")
        else:
          f.write("")

      img_id += 1



def only_patch_images(img, file_name, patch_size=1792):

  if img is None: return

  #-- Configuration --#
  img = pad_patch(img, patch_size) ## pad the image
  h, w, _ = img.shape
  patches_h = h//patch_size
  patches_w = w//patch_size
  img_id = 0
  obj_id = 0


  #-- Splitting Images --#
  for idx_1, h_idx in enumerate(range(patches_h)): #
    for idx_2, w_idx in enumerate(range(patches_w)): #

      # calculate patch
      x = h_idx * patch_size
      y = w_idx * patch_size

      # create patch (patch_size)
      patch = img[x:x+patch_size, y:y+patch_size]

      # # writing paths (patch, labels)
      patch_path = os.path.join(output_image_dir, f"{file_name[:-4]}_patch_{h_idx}_{w_idx}.jpg")

      # write patch & labels
      cv2.imwrite(patch_path, patch)

In [ ]:
# patching large images into slices of 1792x1792 -- only images

import pandas as pd
for idx, csv_file in enumerate(csv_files):

  ## read csv and image
  csv_read_path = os.path.join(root_dir, csv_file)
  read_img_path = os.path.join(root_dir, csv_file[:-3] + "jpg")
  # img_write_path = os.path.join(write_img_dir, f"output_{idx}.jpg")

  single_df = pd.read_csv(csv_read_path)
  required_cols = single_df[['Symbol', 'Xmin', 'Ymin', 'Xmax', 'Ymax']] # target variables

  list_l = []
  for idx, (symbol, xmin, ymin, xmax, ymax) in enumerate(required_cols.values):
    list_l.append([xmin, ymin, xmax, ymax])


  img = cv2.imread(read_img_path)
  patch_images(img, csv_file, *list_l, patch_size=1792)


  print("Reading files csv:{} - img:{}".format(csv_read_path, read_img_path))

In [ ]:
# verifing large images of patches 1792x1792

import json
import cv2
import os

# input dirs
images_path = "/content/images"
labels_path = "/content/labels"
list_files = os.listdir(images_path)
patch_size = 1792

for idx, file_x in enumerate(list_files):
  img_path = os.path.join(images_path, file_x)
  label_path = os.path.join(labels_path, file_x[:-3] + "txt")


  # read image and labels
  img = cv2.imread(img_path)
  with open(label_path, "r") as f:
    data = f.readlines()

  # convert labels
  labels = [] # hold labels from text
  for line in data:
    x_center, y_center, width, height = line.split(" ")[1:-1]
    labels.append([float(x_center), float(y_center), float(width), float(height)])
  # preprocess labels
  labels = yolo_to_xywh(labels, patch_size, patch_size)
  labels = xywh_to_xyxy(labels)

  if labels is None: continue
  for label_x in labels:
    cv2.rectangle(img, (label_x[0], label_x[1]), (label_x[2], label_x[3]), (0, 255, 0), 2)

  cv2.imwrite(os.path.join(write_img_dir, file_x), img)

In [ ]:
# extract the blank(which has no symbol annotations) images
# it is not usable I wrote it because I was needed it for finding
# those images which have no label or annotated class
images_path = "/content/images"
labels_path = "/content/labels"
empty_label_imgs = []
for idx, file_x in enumerate(os.listdir(labels_path)):
  label_path = os.path.join(labels_path, file_x)


  with open(label_path, "r") as f:
    data = f.readlines()

  flag = False
  if len(data) == 0:
    print(label_path)
    empty_label_imgs.append(file_x)

    flag = True

  # if flag:
  #   break

In [ ]:
!mkdir /content/symbol_dataset

!cp -r /content/images /content/symbol_dataset/
!cp -r /content/labels /content/symbol_dataset/

In [ ]:
!zip -r /content/symbol_dataset.zip /content/symbol_dataset

In [ ]:
!cp /content/symbol_dataset.zip /content/drive/MyDrive/Symbol_Detection/

### Loading Data & Verifying Dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!rm -rf /content/Roboflow\ Output
!rm -rf /content/dataset_01/
!rm -rf /content/dataset_02/
!rm -rf /content/dataset_03/
!rm -rf /content/dataset_04/
!rm -rf /content/dataset_05/
!rm -rf /content/dataset_06/
!rm -rf /content/dataset_07/
!rm -rf /content/dataset_08/
!rm -rf /content/dataset_09/
!rm -rf /content/dataset_10/

In [ ]:
# -- dataset zip files paths --
root_dataset_path_00 = "/content/drive/MyDrive/Symbol_Detection/annotated_data_01/Roboflow\ Output.zip"
annotated_dataset_path_01 = "/content/Roboflow\ Output/My\ First\ Project.v3i.yolov8.zip"
annotated_dataset_path_02 = "/content/Roboflow\ Output/Split_Images.v5i.yolov8.zip"
annotated_dataset_path_03 = "/content/Roboflow\ Output/batch\ 3.zip"
annotated_dataset_path_04 = "/content/Roboflow\ Output/batch-4-raw_annotated_dataset.v3i.yolov8.zip"
annotated_dataset_path_05 = "/content/Roboflow\ Output/Batch\ 5\ -\ raw_annotated_dataset.v3i.yolov8.zip"
annotated_dataset_path_06 = "/content/Roboflow\ Output/Batch\ 6\ -\ raw_annotated_dataset.v3i.yolov8.zip"
annotated_dataset_path_07 = "/content/Roboflow\ Output/Batch\ 7-\ raw_annotated_dataset_05.v3i.yolov8.zip"
annotated_dataset_path_08 = "/content/Roboflow\ Output/batch_8_raw_annotated_dataset.v2i.yolov8.zip"
annotated_dataset_path_09 = "/content/Roboflow\ Output/batch_9_raw_annotated_dataset.v1i.yolov8.zip"
annotated_dataset_path_10 = "/content/drive/MyDrive/Symbol_Detection/annotated_data_01/raw_empty_data.zip"




# -- unzip dataset --
!unzip -q {root_dataset_path_00}
!unzip -q {annotated_dataset_path_01} -d dataset_01
!unzip -q {annotated_dataset_path_02} -d dataset_02
!unzip -q {annotated_dataset_path_03} -d dataset_03
!unzip -q {annotated_dataset_path_04} -d dataset_04
!unzip -q {annotated_dataset_path_05} -d dataset_05
!unzip -q {annotated_dataset_path_06} -d dataset_06
!unzip -q {annotated_dataset_path_07} -d dataset_07
!unzip -q {annotated_dataset_path_08} -d dataset_08
!unzip -q {annotated_dataset_path_09} -d dataset_09
!unzip -q {annotated_dataset_path_10} -d dataset_10

In [ ]:
# -- batch 3 sub zip files --
annotated_dataset_path_03_01 = "/content/dataset_03/batch\ 3/Sonam\ Z1.v3i.yolov8.zip"
annotated_dataset_path_03_02 = "/content/dataset_03/batch\ 3/Sonam\ Z2.v3i.yolov8.zip"
annotated_dataset_path_03_03 = "/content/dataset_03/batch\ 3/Tshering.v3i.yolov8.zip"


# -- unzip dataset --
!unzip -q {annotated_dataset_path_03_01} -d dataset_03_01
!unzip -q {annotated_dataset_path_03_02} -d dataset_03_02
!unzip -q {annotated_dataset_path_03_03} -d dataset_03_03

In [ ]:
!rm -rf /content/dataset_03_01
!rm -rf /content/dataset_03_02
!rm -rf /content/dataset_03_03

In [ ]:
-> Symbol ->  400
-> Background -> 20

In [ ]:
# this script actually coping background images for
# blank class of detection dataset to make it
# sutaible for processing

import shutil
import os

input_root_img = "/content/dataset_10/content/yolov9/empty_images/images/"
input_root_lbl = "/content/dataset_10/content/yolov9/empty_images/labels/"
output_root_img = "/content/dataset_10/train/images"
output_root_lbl = "/content/dataset_10/train/labels/"

os.makedirs(output_root_img, exist_ok=True)
os.makedirs(output_root_lbl, exist_ok=True)

for idx, img in enumerate(os.listdir(input_root_img)):
  src_img = os.path.join(input_root_img, img)
  dst_img = os.path.join(output_root_img, img)
  src_lbl = os.path.join(input_root_lbl, img[:-3]+"txt")
  dst_lbl = os.path.join(output_root_lbl, img[:-3]+"txt")

  shutil.move(src_img, dst_img)
  shutil.move(src_lbl, dst_lbl)

  if idx==20: break

In [ ]:
len(os.listdir(output_root_lbl)), len(os.listdir(output_root_img))

(1, 1)

In [ ]:
!rm -rf /content/output_images

In [ ]:
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values

import os
import cv2

images_path = "/content/symbol_detection/val/images"
labels_path = "/content/symbol_detection/val/labels"
write_img_dir = "/content/output_images"
patch_size = 1792  # Not directly used unless you want to draw grid or check sizes

os.makedirs(write_img_dir, exist_ok=True)
list_files = os.listdir(images_path)

for img_file in os.listdir(images_path):
  if not img_file.endswith(('.jpg', '.png', '.jpeg')):
      continue

  image_path = os.path.join(images_path, img_file)
  label_path = os.path.join(labels_path, img_file.rsplit('.', 1)[0] + ".txt")

  img = cv2.imread(image_path)
  if img is None:
    print(f"Could not read image: {image_path}")
    continue
  h, w = img.shape[:2]
  if os.path.exists(label_path):
    with open(label_path, 'r') as f:
      lines = f.readlines()

    for line in lines:
      parts = line.strip().split()
      if len(parts) != 5:
        continue
      class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)
      # Denormalize
      x_center *= w
      y_center *= h
      bbox_width *= w
      bbox_height *= h
      x1 = int(x_center - bbox_width / 2)
      y1 = int(y_center - bbox_height / 2)
      x2 = int(x_center + bbox_width / 2)
      y2 = int(y_center + bbox_height / 2)

      # Draw rectangle and class label
      cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 1)
      # cv2.putText(img, str(int(class_id)), (x1, y1 - 10),
      #             cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

  output_path = os.path.join(write_img_dir, img_file)
  cv2.imwrite(output_path, img)

### Split dataset into Training & Validation

In [ ]:
import numpy as np
import glob
import os
import shutil


# -- dataset paths --
datasets = [
    ("/content/dataset_01/train/images", "/content/dataset_01/train/labels"),
    ("/content/dataset_02/train/images", "/content/dataset_02/train/labels"),
    ("/content/dataset_04/train/images", "/content/dataset_04/train/labels"),
    ("/content/dataset_05/train/images", "/content/dataset_05/train/labels"),
    ("/content/dataset_06/train/images", "/content/dataset_06/train/labels"),
    ("/content/dataset_07/train/images", "/content/dataset_07/train/labels"),
    ("/content/dataset_08/train/images", "/content/dataset_08/train/labels"),
    ("/content/dataset_09/train/images", "/content/dataset_09/train/labels"),
    ("/content/dataset_10/train/images", "/content/dataset_10/train/labels"),
    ("/content/dataset_03_01/train/images", "/content/dataset_03_01/train/labels"),
    ("/content/dataset_03_02/train/images", "/content/dataset_03_02/train/labels"),
    ("/content/dataset_03_03/train/images", "/content/dataset_03_03/train/labels"),
]



# -- new dataset configuration --
root_dir = "symbol_detection"
train_images = f"{root_dir}/train/images"
train_labels = f"{root_dir}/train/labels"
val_images = f"{root_dir}/val/images"
val_labels = f"{root_dir}/val/labels"



# -- make directories --
os.makedirs(train_images, exist_ok=True)
os.makedirs(train_labels, exist_ok=True)
os.makedirs(val_images, exist_ok=True)
os.makedirs(val_labels, exist_ok=True)



# -- move data to train directory --
for img_dir, label_dir in datasets:
    for img_file in glob.glob(f"{img_dir}/*"):
        shutil.move(img_file, train_images)
    for label_file in glob.glob(f"{label_dir}/*"):
        shutil.move(label_file, train_labels)



# -- list files --
raw_images = os.listdir(train_images)
raw_labels = os.listdir(train_labels)
raw_images.sort()
raw_labels.sort()



# -- Separate Samples For Validation --
np.random.seed(42)
num_samples = int(len(raw_images) * 0.04) # 4 per%
selected_indices = np.random.choice(len(raw_images), size=num_samples, replace=False)
random_raw_valid_images = [raw_images[i] for i in selected_indices]
random_raw_valid_labels = [raw_labels[i] for i in selected_indices]



# -- Move data to Validation directories --
for i in range(len(random_raw_valid_images)):
    src_img_path = os.path.join(train_images, random_raw_valid_images[i])
    dst_img_path = os.path.join(val_images, random_raw_valid_images[i])
    src_lbl_path = os.path.join(train_labels, random_raw_valid_labels[i])
    dst_lbl_path = os.path.join(val_labels, random_raw_valid_labels[i])

    shutil.move(src_img_path, dst_img_path)
    shutil.move(src_lbl_path, dst_lbl_path)


print("Training Samples: Images: {} - Labels {}".format(len(os.listdir(train_images)), len(os.listdir(train_labels))))
print("Validation Samples: Images: {} - Labels {}".format(len(os.listdir(val_images)), len(os.listdir(val_labels))))

Training Samples: Images: 4176 - Labels 4176
Validation Samples: Images: 174 - Labels 174


In [ ]:
# found certain annotations issues
# which need to be handled to not get
# the model bias during training

# -- training data --
for file_x in os.listdir(train_labels):
    adj_label_path = os.path.join(train_labels, file_x)
    # Collect valid lines
    write_flag = False
    valid_lines = []
    with open(adj_label_path, "r") as read_file:
        for line in read_file:
            data = line.strip().split()
            try:
                class_id, x_center, y_center, bbox_w, bbox_h = map(float, data)
                valid_lines.append(f"{int(class_id)} {x_center} {y_center} {bbox_w} {bbox_h}")
            except ValueError:
                write_flag = True
                continue
    # Overwrite file with only valid lines
    if write_flag:
      with open(adj_label_path, "w") as write_f:
          for line in valid_lines:
              write_f.write(line + "\n")
print("Irregular Labels Fixed For Train Dataset Completed Successfully")


# -- validation data --
# for file_x in os.listdir(train_labels):
for file_x in os.listdir(val_labels):
    adj_label_path = os.path.join(val_labels, file_x)
    # Collect valid lines
    write_flag = False
    valid_lines = []
    with open(adj_label_path, "r") as read_file:
        for line in read_file:
            data = line.strip().split()
            try:
                class_id, x_center, y_center, bbox_w, bbox_h = map(float, data)
                valid_lines.append(f"{int(class_id)} {x_center} {y_center} {bbox_w} {bbox_h}")
            except ValueError:
                write_flag = True
                continue
    # Overwrite file with only valid lines
    if write_flag:
      with open(adj_label_path, "w") as write_f:
          for line in valid_lines:
              write_f.write(line + "\n")
print("Irregular Labels Fixed For Valid Dataset Completed Successfully")

Irregular Labels Fixed For Train Dataset Completed Successfully
Irregular Labels Fixed For Valid Dataset Completed Successfully


In [ ]:

# -- train dataset --
for file_x in os.listdir(train_labels):
  adj_label_path = os.path.join(train_labels, file_x)
  file_lines = []
  with open(adj_label_path, "r") as read_file:
    for line in read_file:
      data = line.strip().split()
      class_id, x_center, y_center, bbox_w, bbox_h = map(float, data)
      file_lines.append(f"{int(0)} {x_center} {y_center} {bbox_w} {bbox_h}")
      # print("class_id {} - x_cen {} - y_cen {} - w {} - h {}".format(class_id, x_center, y_center, bbox_w, bbox_h))

  with open(adj_label_path, "w") as write_f:
      for line in file_lines:
          write_f.write(line + "\n")
print("Class Of Symbol Successfully Set To 0 For Training Dataset")



# -- valid dataset --
for file_x in os.listdir(val_labels):
  adj_label_path = os.path.join(val_labels, file_x)
  file_lines = []
  with open(adj_label_path, "r") as read_file:
    for line in read_file:
      data = line.strip().split()
      class_id, x_center, y_center, bbox_w, bbox_h = map(float, data)
      file_lines.append(f"{int(0)} {x_center} {y_center} {bbox_w} {bbox_h}")
      # print("class_id {} - x_cen {} - y_cen {} - w {} - h {}".format(class_id, x_center, y_center, bbox_w, bbox_h))

  with open(adj_label_path, "w") as write_f:
      for line in file_lines:
          write_f.write(line + "\n")
print("Class Of Symbol Successfully Set To 0 For Validation Dataset")

Class Of Symbol Successfully Set To 0 For Training Dataset
Class Of Symbol Successfully Set To 0 For Validation Dataset


In [ ]:
# -- train dataset --
for file_x in os.listdir(train_labels):
  adj_label_path = os.path.join(train_labels, file_x)
  file_lines = []
  with open(adj_label_path, "r") as read_file:
    for line in read_file:
      data = line.strip().split()
      class_id, x_center, y_center, bbox_w, bbox_h = map(float, data)
      # file_lines.append(f"{int(0)} {x_center} {y_center} {bbox_w} {bbox_h}")
      print("class_id {} - x_cen {} - y_cen {} - w {} - h {}".format(class_id, x_center, y_center, bbox_w, bbox_h))

### Data augmentation

#### augment_01: 90-degree clockwise rotation

In [ ]:
import os
import cv2
import numpy as np
import albumentations as A


# augmentation (90-degree rotation)
transform = A.Compose([
  A.Rotate(limit=[90, 90], p=1)
])

def correct_yolo_labels_after_rotation(label_path, img_width, img_height):
  """ Adjust YOLO bounding boxes after a 90-degree clockwise rotation. """
  new_labels = []
  with open(label_path, "r") as file:
    lines = file.readlines()
  for line in lines:
    data = line.strip().split()
    class_id, x_center, y_center, bbox_w, bbox_h = map(float, data)

    # Convert to absolute values
    x_center_abs = x_center * img_width
    y_center_abs = y_center * img_height
    bbox_w_abs = bbox_w * img_width
    bbox_h_abs = bbox_h * img_height

    # Transform coordinates for 90-degree rotation
    new_x_center_abs = y_center_abs
    new_y_center_abs = img_width - x_center_abs
    new_bbox_w_abs, new_bbox_h_abs = bbox_h_abs, bbox_w_abs  # Swap width & height

    # Normalize back to YOLO format
    new_x_center = new_x_center_abs / img_height
    new_y_center = new_y_center_abs / img_width
    new_bbox_w = new_bbox_w_abs / img_height
    new_bbox_h = new_bbox_h_abs / img_width

    new_labels.append(f"{int(class_id)} {new_x_center} {new_y_center} {new_bbox_w} {new_bbox_h}\n")
  return new_labels



# Process all images
for img_file in os.listdir(image_dir):
  if img_file.endswith(".jpg") or img_file.endswith(".png"):
    img_path = os.path.join(image_dir, img_file)
    label_path = os.path.join(label_dir, img_file.replace(".jpg", ".txt").replace(".png", ".txt"))

    # Load Image
    image = cv2.imread(img_path)
    img_height, img_width = image.shape[:2]
    # Rotate Image
    augmented = transform(image=image)
    rotated_image = augmented["image"]
    # Save rotated image
    rotated_img_path = os.path.join(output_image_dir, f"aug_0_{img_file}")
    cv2.imwrite(rotated_img_path, rotated_image)
    # Adjust & Save YOLO labels
    if os.path.exists(label_path):
      try:
        new_labels = correct_yolo_labels_after_rotation(label_path, img_width, img_height)
      except Exception as e:
        print(f"Error processing {img_file}: {e}")
        continue
      rotated_label_path = os.path.join(output_label_dir, f"aug_0_{img_file.replace('.jpg', '.txt').replace('.png', '.txt')}")

      with open(rotated_label_path, "w") as file:
          file.writelines(new_labels)
    print(f"Processed: {img_file}")

if __name__=="__main__":
  # Input & Output Directories
  image_dir = "/content/symbol_detection/train/images"  # Folder with original images
  label_dir = "/content/symbol_detection/train/labels"  # Folder with YOLO labels
  output_image_dir = "/content/symbol_detection/train/images"  # Save rotated images
  output_label_dir = "/content/symbol_detection/train/labels"  # Save updated labels

  # output directories exist
  os.makedirs(output_image_dir, exist_ok=True)
  os.makedirs(output_label_dir, exist_ok=True)


  print("Augmentation completed for all images.")

In [ ]:
print(len(os.listdir(train_images)))
print(len(os.listdir(train_labels)))

#### augment_02: - color - noise - blurness

In [ ]:
from tqdm import tqdm

# -- augmentation pipeline --
transform = A.Compose([
    A.RandomBrightnessContrast(
     brightness_limit=0.2,
     contrast_limit=0.2,
     brightness_by_max=False,
     p=0.3
    ),

    A.GaussianBlur(blur_limit=3, p=0.3),  # Light Gaussian blur
    A.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1, p=0.4),
    A.RandomGamma(p=1),               # Adjust gamma
    A.RandomFog(fog_coef_lower=1, fog_coef_upper=1, p=0.3),
    A.RandomGamma(gamma_limit=(80, 120), p=0.5),
])
# bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']),


# -- Paths setup --
input_images = "/content/symbol_detection/train/images"     # Original images folder
input_labels = "/content/symbol_detection/train/labels"     # Original Labels folder
output_images = "/content/symbol_detection/train/images"   # Augmented images folder
output_labels = "/content/symbol_detection/train/labels"   # Augmented labels folder

os.makedirs(output_images, exist_ok=True)
os.makedirs(output_labels, exist_ok=True)


# -- Process images & labels --
augment = 1
for image_file in tqdm(os.listdir(input_images)):
  if image_file.endswith(('.png', '.jpg', '.jpeg')):
    img_path = os.path.join(input_images, image_file)
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    if img is not None:
      augmented = transform(image=img)["image"]
      # Save augmented image with a new name
      output_img_file_path = f"aug_{augment}_{image_file}"
      output_lbl_file_path = f"aug_{augment}_{image_file[:-3]}"+"txt"
      output_img_path = os.path.join(output_images, output_img_file_path)
      output_lbl_path = os.path.join(output_labels, output_lbl_file_path)
      with open(os.path.join(input_labels, image_file[:-3]+"txt"), 'r') as infile:
        content = infile.read()
      cv2.imwrite(output_img_path, augmented)
      with open(output_lbl_path, "w") as outfile:
        outfile.write(content)


print("\n\n")
print(f"original examples    :  {len(os.listdir(input_images))//2}")
print(f"augmented examples   :  {len(os.listdir(output_images))//2}")
print(f"total examples       :  {len(os.listdir(input_images))//2 + len(os.listdir(output_labels))//2}")

In [ ]:
# zip the dataset
import zipfile

# -- folder setup --
source_folder = "/content/dataset_v2/"
zip_filename = "/content/custom-dataset-fractiion-imporovement.zip"

# -- zip the files --
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(source_folder):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, source_folder)
            zipf.write(file_path, arcname=arcname)

print(f"All files in '{source_folder}' have been zipped into '{zip_filename}'.")
print(f"Size of zip file is {os.path.getsize(zip_filename)} bytes")

In [ ]:
# copy the zip dataset into drive

!cp /content/custom-dataset-fractiion-imporovement.zip /content/drive/MyDrive/OCR-Custom-Dataset/

## Training Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# clone the repo for training and testing
!git clone https://github.com/SkalskiP/yolov9.git
%cd yolov9

!pip install -r requirements.txt -q
%cd /content/

Cloning into 'yolov9'...
remote: Enumerating objects: 325, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 325 (delta 145), reused 142 (delta 142), pack-reused 135 (from 1)
Receiving objects: 100% (325/325), 2.23 MiB | 4.04 MiB/s, done.
Resolving deltas: 100% (165/165), done.
/content/yolov9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00


In [ ]:
# download pretrained weights (imagenet trained)

!wget -P weights -q https://github.com/WongKinYiu/yolov9/releases/download/v0.1/gelan-c.pt
!wget -P weights -q https://github.com/WongKinYiu/yolov9/releases/download/v0.1/gelan-e.pt

In [ ]:
# copy the weights

!mkdir /content/weights/
!cp -r /content/drive/MyDrive/Symbol_Detection/models/* /content/weights/
!ls -la /content/weights/

total 911952
drwxr-xr-x 2 root root      4096 Aug  3 12:21 .
drwxr-xr-x 1 root root      4096 Aug  3 12:21 ..
-rw------- 1 root root 466907182 Aug  3 12:21 best_aug_02.pt
-rw------- 1 root root 466907182 Aug  3 12:22 best_july_19.pt


### Datatset Format

- Yolo Format Dataset
  - dataset
      - train
        - images
        - labels
      - val
        - images
        - labels



- The Image and labels file name should be same
- The images to be store within the images directory (img.jpg)
- labels format (img.txt)
  - each labels should have the following structure
  - class, xc, yc, w, h      <---  Normalized values

In [ ]:
import os

# -- dataset dirs structure --
root_dataset_dir = "/content/symbol_detection"
root_train_dir = "/content/symbol_detection/train"
root_val_dir = "/content/symbol_detection/val"



print("Training Images & Labels")
print(len(os.listdir(f"{root_train_dir}/images")))
print(len(os.listdir(f"{root_train_dir}/labels")))


print("\nValidation Images & Labels")
print(len(os.listdir(f"{root_val_dir}/images")))
print(len(os.listdir(f"{root_val_dir}/labels")))

Training Images & Labels
4176
4176

Validation Images & Labels
174
174


In [ ]:
import yaml

data = {
    'names': ['Symbol'],
    'nc': 1,
    'train': './symbol_detection/train/images',
    'val': './symbol_detection/val/images'
}

with open('/content/symbol_detection/data.yaml', 'w') as file:
    yaml.dump(data, file, sort_keys=False)

In [ ]:
# move/copy the symbol_detection dataset into yolov9 directory

!mv /content/symbol_detection/ /content/yolov9

In [ ]:
# start training the model
%cd /content/yolov9

!python train.py \
--batch 3 --epochs 60 --img 1792 --min-items 0 --workers 0 --close-mosaic 13 \
--data /content/yolov9/symbol_detection/data.yaml \
--weights /content/weights/gelan-e.pt \
--cfg models/detect/gelan-e.yaml \
--hyp hyp.scratch-high.yaml

/content/yolov9
2025-08-01 18:57:52.126443: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-01 18:57:52.144005: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754074672.165626    1726 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754074672.172080    1726 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754074672.189399    1726 computation_placer.cc:177] computation placer already registered. Please check linkage an

In [ ]:
!cp /content/yolov9/runs/train/exp/weights/best.pt /content/drive/MyDrive/Symbol_Detection/models/

## Testing

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# unzip the test data

!unzip -q /content/drive/MyDrive/Symbol_Detection/Batch\ 9\ -\ Legends.zip -d raw_data

In [ ]:
!pip install -q PyMuPDF Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 67.6 MB/s eta 0:00:00


In [ ]:
# Install in Colab:

import fitz               # PyMuPDF
from PIL import Image, ImageFilter
import os
import shutil

def pdf_to_jpg(pdf_path, output_folder="output_images", dpi=300,
               threshold=220, output_folder_bw="bw_imgs", pdf_id=0, blur_radius=0.5):
    """
    Convert each page of a PDF to:
      • color_image_<page>.jpg
      • bw_image_<page>.jpg

    Args:
      pdf_path (str): path to .pdf file
      output_folder (str): where to save JPEGs
      dpi (int): rendering resolution
      threshold (0–255): gray cutoff for B/W
      blur_radius (float): Gaussian blur radius on B/W
    """
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(output_folder_bw, exist_ok=True)
    doc = fitz.open(pdf_path)
    page_count = doc.page_count

    for i, page in enumerate(doc, start=1):
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # — Color JPEG
        color_file = os.path.join(output_folder, f"color_image_{i}.jpg")
        img.save(color_file, "JPEG", quality=100, dpi=(dpi, dpi))

        # — Grayscale → threshold → blur → B/W JPEG
        gray = img.convert("L")
        bw = gray.point(lambda x: 255 if x > threshold else 0)
        bw = bw.filter(ImageFilter.GaussianBlur(blur_radius))
        bw_file = os.path.join(output_folder_bw, f"bw_image_{pdf_id}_{i}.jpg")
        bw.save(bw_file, "JPEG", quality=100, dpi=(dpi, dpi))

    doc.close()
    print(f"Done! For pdf {pdf_id} Saved {page_count} page(s) to '{output_folder}'")

if __name__ == "__main__":
    # Adjust to your Colab PDF path
      # pdf_path = "/content/90-QP20-R-001_1_Process & Inst_Diagram _PID_Legend_and_Sybmols.pdf"
      # pdf_to_jpg(pdf_path,
      #            output_folder="/content/output_images")



    input_dir = "/content/testing"
    output_bw = "/content/bw_images/"
    os.makedirs(output_bw, exist_ok=True)
    for idx, file_x in enumerate(os.listdir(input_dir)):

      # pdf
      if (file_x.endswith(".pdf") or file_x.endswith(".PDF")):
        pdf_path = os.path.join(input_dir, file_x)
        pdf_to_jpg(pdf_path,
                  output_folder="/content/output_images",
                  pdf_id = idx,
                  output_folder_bw=output_bw)

      # image
      elif (file_x.endswith(".jpg") or file_x.endswith(".JPG")):
        # Corrected destination path for shutil.copy
        src_path = os.path.join(input_dir, file_x)
        dst_path = os.path.join(output_bw, file_x)
        print(src_path, dst_path)
        shutil.copy(src_path, dst_path)
        print(f"Done! For {file_x} Copied {dst_path}")


      # skip
      else:
        print(f"Skipping {file_x}")

Done! For pdf 0 Saved 1 page(s) to '/content/output_images'
Done! For pdf 1 Saved 1 page(s) to '/content/output_images'


In [ ]:
# code for creating overlapping patches

import os
import math
import cv2
import numpy as np

def pad_patch(img, patch_size, overlap_ratio=0.3):
  """Pad the image so that it fits evenly into overlapping patches

  Args:
    img (np.array): input image (HWC)
    patch_size (int): size of the square patch
    overlap_ratio (float): overlap between patches (e.g., 0.3 for 30%)

  Returns:
    padded_img (np.array): padded image
    stride (int): stride based on overlap
  """
  h, w = img.shape[:2]
  stride = int(patch_size * (1 - overlap_ratio))

  # Compute how many patches we need in each dimension
  num_patches_h = math.ceil((h - patch_size) / stride) + 1
  num_patches_w = math.ceil((w - patch_size) / stride) + 1

  # Final padded size
  final_h = stride * (num_patches_h - 1) + patch_size
  final_w = stride * (num_patches_w - 1) + patch_size

  pad_h = final_h - h
  pad_w = final_w - w

  padded_img = cv2.copyMakeBorder(
    img,
    0, pad_h,
    0, pad_w,
    cv2.BORDER_CONSTANT,
    value=(255, 255, 255)
  )
  return padded_img, stride


def only_patch_images(img, file_name, output_image_dir, patch_size=1792, overlap_ratio=0.3):
  """Generate overlapping patches from image

  Args:
    img (np.array): input image
    file_name (str): name of the original file
    output_image_dir (str): directory to save patches
    patch_size (int): patch size
    overlap_ratio (float): e.g. 0.3 for 30% overlap
  """
  if img is None: return

  # Pad and get stride
  img, stride = pad_patch(img, patch_size, overlap_ratio)
  h, w, _ = img.shape
  patch_id = 0

  # Create patches with overlap
  for y in range(0, h - patch_size + 1, stride):
    for x in range(0, w - patch_size + 1, stride):
      patch = img[y:y + patch_size, x:x + patch_size]

      patch_path = os.path.join(
        output_image_dir,
        f"{file_name[:-4]}_patch_{y}_{x}.jpg"
      )
      cv2.imwrite(patch_path, patch)
      patch_id += 1


if __name__=='__main__':

  input_image_dir = "/content/bw_images"
  output_dir = "/content/overlap_patches"
  os.makedirs(output_dir, exist_ok=True)

  for idx, img_name in enumerate(os.listdir(input_image_dir)):
    diagram_img_path = os.path.join(input_image_dir, img_name)
    image = cv2.imread(diagram_img_path)
    only_patch_images(image, img_name, output_dir,
                      patch_size=1792, overlap_ratio=0.3)

    print("Reading files {} - img:{}".format(idx, diagram_img_path))

Reading files 0 - img:/content/bw_images/bw_image_0_1.jpg
Reading files 1 - img:/content/bw_images/bw_image_1_1.jpg


In [ ]:
%cd /content/yolov9
!python detect.py \
--img 1792 --conf 0.3 \
--weights /content/weights/best_aug_02.pt \
--source /content/overlap_patches

/content/yolov9
detect: weights=['/content/yolov9/runs/train/exp/weights/best.pt'], source=/content/overlap_patches, data=data/coco128.yaml, imgsz=[1792, 1792], conf_thres=0.3, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=2, hide_labels=True, hide_conf=True, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 1e33dbb Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)

Fusing layers... 
gelan-e summary: 930 layers, 58002835 parameters, 0 gradients, 190.8 GFLOPs
image 1/36 /content/overlap_patches/bw_image_0_1_patch_0_0.jpg: 1792x1792 5 Symbols, 67.9ms
image 2/36 /content/overlap_patches/bw_image_0_1_patch_0_1254.jpg: 1792x1792 10 Symbols, 58.5ms
image 3/36 /content/overlap_patches/bw_image_0_1_patch_0_2508.jpg: 1792x1792 6 Symbols, 58.4ms
image 4/36 /content/

In [ ]:
!mkdir /content/dataset_images/


!cp -r /content/patch_images/* /content/dataset_images/
!cp -r /content/yolov9/runs/detect/exp2/labels/* /content/dataset_images/

In [ ]:
!cp -r /content/yolov9/runs/train/exp/weights/best.pt /content/drive/MyDrive/Symbol_Detection/models/

In [ ]:
!unzip /content/drive/MyDrive/Symbol_Detection/test_results.zip